In [2]:
import pandas as pd

llm = pd.read_parquet('./llm_data/llm_data.parquet')
ds  = pd.read_parquet('./token_counts.parquet')
llm

,key,context,params,in,out,mmlu
id,,,,,,
from-scratch,xaviergillard/brabant-xvii-from-scratch,512.0,1.050000e+08,NaN,NaN,NaN
gpt2,openai-community/gpt2,1024.0,1.370000e+08,NaN,NaN,NaN
gpt3.5-turbo,None,16000.0,1.750000e+11,1.50,2.00,70.00
gpt4-turbo,None,128000.0,NaN,60.00,120.00,86.40
gpt4o,None,128000.0,NaN,5.00,15.00,88.70
gpt4o-mini,None,128000.0,NaN,0.15,0.60,82.00
gysbertv2,xaviergillard/brabant-xvii-shallow,512.0,1.090000e+08,NaN,NaN,NaN
llama3.1-405b,meta-llama/Meta-Llama-3.1-405B-Instruct,128000.0,4.050000e+11,NaN,NaN,88.60
llama3.1-70b,meta-llama/Meta-Llama-3.1-70B-Instruct,128000.0,7.006000e+10,NaN,NaN,86.00


In [10]:
llm['fits_a100_80'] = (llm['params'] * 2 / 1024**3) < 80

In [11]:
llm['generative'] = True
llm.loc[['from-scratch', 'gysbertv2', 'xlm-roberta'], 'generative'] = False
llm.to_parquet('./llm_data/llm_data.parquet', compression='gzip')

In [12]:
must = llm[llm['generative'] & llm['fits_a100_80'] & llm['key']]
must

,key,context,params,in,out,mmlu,fits_a100_80,generative
id,,,,,,,,
gpt2,openai-community/gpt2,1024.0,1.370000e+08,NaN,NaN,NaN,True,True
llama3.1-8b,meta-llama/Meta-Llama-3.1-8B-Instruct,128000.0,8.030000e+09,NaN,NaN,73.0,True,True
mistral-7b,mistralai/Mistral-7B-Instruct-v0.3,32000.0,7.250000e+09,0.20,0.20,62.5,True,True
mistral-nemo,mistralai/Mistral-Nemo-Instruct-2407,128000.0,1.220000e+10,0.27,0.27,68.0,True,True


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

id        = "mistral-7b"
key       = 'mistralai/Mistral-7B-Instruct-v0.3'
text      = ds[ds[id] == ds[id].max()][id].item()

tokenizer = AutoTokenizer.from_pretrained(key)
model     = AutoModelForCausalLM.from_pretrained(key)

inputs    = tokenizer(text, return_tensors='pt')
outputs   = model.generate(**inputs)

    
print(tokenizer.batch_decode(outputs))

In [21]:
from transformers import pipeline

pipe = pipeline('text-generation', model=key, max_new_tokens=500)

print(pipe("Hi xavier how are you doing ?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'Hi xavier how are you doing ? are you going to tell the others how well you are doing? xavier it will be interesting to watch them. can you confirm for me you are doing well on your book. in addition there will be a few small things going in the book, for what i will say at the beginning it may be hard for you to go through everything in this book. i could explain how some people found they could not handle the book, i think the biggest one might be at the end it is the first time we meet people that were not very experienced with the game. im sure you guys would also like to know what you have heard about some of the things i have been asked out for in writing. do you have any comments or any ideas about how you will improve your writing or can you tell me a little about yourself, i will start the book with i write it for every reader who reads here. what do you think are the main problems that affect you since the start and after, as far as the game becomes? and 

In [22]:
inputs

{'input_ids': tensor([[ 197,  198, 9503,  ..., 1452, 8159, 1000]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}

In [40]:
ds[ds['gpt2'] == ds['gpt2'].min()]['text'].item()

'philips etc. allen etc. saluit alsoe als tonsen kennisse comen es hoe dat\nonlancx leden herbrecht vander edelbampt in wille was ende\nbestaen hadde in onser stat van tricht eene jonge maecht\nte ontschaken ende jegens hoiren danck te ontvoeren die\nhem ter stont weder om ontweldicht wairt dair om die\nburgermeesteren gesworen ende raid oner voirs. stat van\ntricht of hoire momboiren van hoiren name den selven\nherbrecht beclaicht ende verwonnen hebben hoe wail die partien\nvrunden ende magen vander voirs. jonger maecht dair af niet\ngeclaicht en hebben ende is de selve herbrecht hem dair om\nhoudende buten onser voirs. stat ende lande van brabant tot zyne\ngroter schaden ende meer zyn soude wairt sake dat wy onse\ngracie ende genade an hem niet en keerden, so doen wy te\nweten dat wy ter liefden vanden eerwairdigen vader in gode\nonsen lieven neve den biscop van ludick die ons dair om heeft\ndoen bidden ende versueken hebben den selven herbrecht vergeven\nende quytgesconden ende met 